# 이미지 하나로 통합 결과 보기
아래 **IMAGE_FILENAME**의 파일명만 바꾸고 셀을 실행하세요.

서버의 TUM 이미지 → PathGen caption → MedGemma 쉬운 설명 + MUSK 5항목 예측.
GPU 작업은 Slurm으로 자동 제출되며, 대기·모델 로딩 중 셀은 실행 상태로 유지됩니다. 완료되면 최종 결과만 표시합니다.

**Delta에 SSH로 연결한 VS Code/Jupyter의 원격 Python 커널**에서 실행하세요. PathGen 설치, MedGemma 기본 가중치·어댑터, 기존 MUSK 학습 모델이 준비돼 있어야 합니다.
기본 모델: `../musk/runs/head_slurm_22984898/best.pt`. 기존 MedGemma 지침대로 설명은 영어입니다.
셀을 정상 중단하면 제출한 작업도 취소 요청합니다. 원본 로그·결과 파일은 서버에 남습니다.


In [ ]:
# 이 파일명만 바꾸고 셀을 실행하세요.
IMAGE_FILENAME = "TUM-YHTSLLSL.tif"

from pathlib import Path
import sys

# 노트북을 connect/ 또는 저장소 루트에서 실행할 수 있습니다.
_connect = None
for _parent in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    for _candidate in (_parent, _parent / "connect"):
        if (_candidate / "notebook_runner.py").is_file():
            _connect = _candidate
            break
    if _connect is not None:
        break
if _connect is None:
    raise RuntimeError("원격 서버의 connect 폴더를 열고 노트북을 실행하세요.")
if str(_connect) not in sys.path:
    sys.path.insert(0, str(_connect))

from notebook_runner import Settings, run_report

# 기존 학습 모델과 서버 환경은 자동 지정됩니다.
# 모델 변경 시 아래 Settings의 checkpoint 옵션에 새 절대경로를 지정하세요.
settings = Settings(connect=_connect)
try:
    result = run_report(IMAGE_FILENAME, settings)
except KeyboardInterrupt:
    print("작업 취소를 요청했습니다.")
except Exception as error:
    print(f"실행 실패: {error}")
else:
    print(result["report"])
